# Research questions II — convolutional networks

Same rules as [notebook 01](01_research_questions.ipynb): pick a question, answer it
experimentally in two to five days, report accuracy **and** FLOPs.

| baseline (`run_mnist(baseline_cnn)`) | test accuracy | train GFLOP | inference GFLOP |
|---|---|---|---|
| q09 CNN, 5×5 kernels, 5 994 params | 0.9713 ± 0.0040 | 79.0 | 6.62 |
| commonest digit | 0.1135 | 0 | 0 |

## Protocol

- Notebook 01's protocol applies: ≥ 3 seeds, mean ± std, cost beside every score.
- A `Conv2d` costs $2F(K+1)NB$ FLOPs forward; shifts, padding and crops cost **0**.
- Report parameters and wall time beside FLOPs.

In [ ]:
import pathlib
import sys

ROOT = pathlib.Path.cwd()
if not (ROOT / "dlcb").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np

from dlcb import engine as cpu
from dlcb import nn
from dlcb.conv import Conv2d, Flatten, MaxPool2d
from research.benchmark import (DigitCNN, baseline_cnn, count_flops, pareto, repeat,
                                run_mnist, summarize)

Every answer has this shape: baseline and variation, three seeds, one table (about 10 minutes).

In [ ]:
rows = []
rows += repeat(lambda s: run_mnist(baseline_cnn, seed=s, name="baseline 5×5"))
rows += repeat(lambda s: run_mnist(lambda: DigitCNN(kernel=9), seed=s, name="kernel 9×9"))
print(summarize(rows, baseline="baseline 5×5"))
pareto(rows, cost="train_flops", title="MNIST: accuracy vs training cost")

## Questions

| # | question | data |
|---|---|---|
| 1 | [Small kernels, large receptive fields](#q1) | MNIST |
| 2 | [Architectural design for noise-robust CNNs](#q2) | MNIST + Gaussian noise |
| 3 | [Self-supervised image denoising](#q3) | MNIST + noise |

<a id="q1"></a>
## 1. Small kernels, large receptive fields

**Question.** Can small kernels combined with spatial shifts match the accuracy of
large-kernel CNNs at a lower computational cost?

**Read**
- D. Li, L. Li, Z. Chen, J. Li. ShiftwiseConv: Small Convolutional Kernel with Large
  Kernel Effect. *Proceedings of the IEEE/CVF Conference on Computer Vision and Pattern
  Recognition (CVPR)*, p. 25281–25291, 2025.
- B. Wu et al. Shift: A Zero FLOP, Zero Parameter Alternative to Spatial Convolutions.
  CVPR 2018. [arXiv:1711.08141](https://arxiv.org/abs/1711.08141)

**Do**
- Implement a simplified `ShiftwiseConv` in `dlcb` (a `shift2d` op, gradient-checked).
- Prove when a sum of shifted $k \times k$ convolutions equals one $K \times K$
  convolution; derive the FLOPs of both and check them against `cpu.flop_count()`.
- Plot accuracy vs FLOPs on MNIST: `DigitCNN(kernel=K)` for $K \in \{5, 7, 9, 11\}$
  against shift-based networks with $k = 3$.

<a id="q2"></a>
## 2. Architectural design for noise-robust CNNs

**Question.** Which architectural features of a CNN favour robustness to noise
distributions not seen during training?

**Read**
- B. J. Kim, M. Kawano, Y. Iwasawa, Y. Matsuo. Unlocking Noise-Resistant Vision: Key
  Architectural Secrets for Robust Models Against Gaussian Noise. *Proceedings of the 43rd
  International Conference on Machine Learning (ICML)*, PMLR, v. 306, p. 57781–57815,
  2026. [proceedings.mlr.press/v306/kim26w](https://proceedings.mlr.press/v306/kim26w.html)

**Do**
- Train on clean digits; test on clean (ID) and Gaussian-noised (OOD) digits,
  $\sigma \in \{0.1, 0.2, 0.4\}$.
- Derive how noise variance propagates through a convolution and through max and average
  pooling, and state the hypotheses it predicts for kernel size, pooling and filter
  properties.
- Vary kernel size, pooling and filter properties; plot ID and OOD accuracy vs FLOPs.

<a id="q3"></a>
## 3. Self-supervised image denoising

**Question.** Can a CNN learn to restore images corrupted by strong noise without ever
seeing a clean image during training?

**Read**
- T. Li, L. Wang, Z. Xu, L. Zhu, W. Lu, H. Huang. Positive2Negative: Breaking the
  Information-Lossy Barrier in Self-Supervised Single Image Denoising. *CVPR*,
  p. 17924–17934, 2025. [doi:10.1109/CVPR52734.2025.01670](https://doi.org/10.1109/CVPR52734.2025.01670)

**Do**
- State and prove the statistical assumptions on the noise under which a loss computed
  from noisy images alone recovers the clean image.
- On MNIST, train a supervised denoiser (noisy → clean) and a self-supervised one (noisy
  images only); measure restoration (PSNR), the accuracy of the q09 classifier on the
  restored digits, ID and OOD noise levels, and FLOPs.

## Hand in

The report template and grading of [notebook 01](01_research_questions.ipynb).